## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240219'
end_date = '20240303'

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v1/city/chennai/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v1
/Users/rapido/local-datasets/customer/appography/appography_v1/city/chennai/


##usecase_tag

usecase_tag_query = f"""

WITH active_customers AS (

    SELECT 
        customer_id,
        order_id,
        drop_location_hex_10 hex_10
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host = 'android'
        AND city_name = 'Chennai'
    GROUP BY 1,2,3
    ), 

    use_case AS (
    
    SELECT
        hex_10, 
        usecase_tag
    FROM
        (
        SELECT 
            hex_10, 
            combined_final_usecase_accuracy as usecase_tag,
            ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
        FROM
            hive.experiments_internal.combined_usecase_hex_10_level
        WHERE 
            aoi = 'Chennai District, India'
        )
    WHERE   
        seq_no = 1
    ),
    
    merge AS (
    SELECT
        customer_id,
        usecase_tag,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY orders DESC) seq_no
    FROM 
        (
        SELECT
            a.customer_id,
            COALESCE(b.usecase_tag, 'Unknown') usecase_tag,
            COUNT(DISTINCT order_id) orders
        FROM 
            active_customers a
        LEFT JOIN 
            use_case b
            ON a.hex_10 = b.hex_10
        GROUP BY 1,2
        )
    WHERE 
        usecase_tag != 'Unknown'
    )
    
    SELECT
        a.customer_id,
        COALESCE(b.usecase_tag, 'Unknown') usecase_tag
    FROM 
        active_customers a
    LEFT JOIN 
        merge b 
        ON a.customer_id = b.customer_id
        AND seq_no = 1
    GROUP BY 1,2

"""

df_usecase_tag_query = pd.read_sql(usecase_tag_query, connection)
df_usecase_tag_query.to_csv(local_datasets + 'raw/usecase_tag.csv', index=False)

In [6]:
df_usecase_tag = pd.read_csv(local_datasets + 'raw/usecase_tag.csv')
print(df_usecase_tag.shape)
df_usecase_tag.head(2)

(812108, 2)


,customer_id,usecase_tag
0,64e5bccdf2e50d14f8f4b9d0,residential
1,613ac8177f5ff7dfbe012276,transit_station


### Active customer (RR-customers)

In [7]:
df_bangalore_active_customer = pd.read_csv(local_datasets + 'raw/chennai_customers.csv')
df_bangalore_active_customer = df_bangalore_active_customer.drop('app_list', axis=1)
print(df_bangalore_active_customer.shape)
df_bangalore_active_customer.head(2)

(484718, 16)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,net_count_with_nan,rpc
0,657bc9b5a55bd4f3ee494ef6,FEMALE,86.0,NaN,27.0,16.0,7.0,PHH,12,COMMITTED,UNKNOWN,ONLY_LINK,SHORT_DISTANCE,PS,7.0,d. high rpc
1,629c2da2a3627e12eb8d4de5,FEMALE,644.0,NaN,20.0,11.0,7.0,PHH,29,COMMITTED,MEDIUM_INCOME,ONLY_AUTO,LONG_DISTANCE,NPS,7.0,d. high rpc


In [8]:
df_bangalore_active_customer = pd.merge(df_bangalore_active_customer, df_usecase_tag,
                                        how='left', on=['customer_id'])

### customer installed apps

In [9]:
df_customer_mapped = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapped.csv')
print(df_customer_mapped.shape)
df_customer_mapped.head(2)

(484284, 13)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Ride haling,Rest,Finance_Investment,Gaming,Driver_App,Educational
0,573f28f39b0ffc283676f1dd,"['axis mobile', 'redbus', 'messenger', 'facebo...",14,"['Ecommerce', 'Finance_Transactions', 'Deliver...",7,['Rest'],1,0,1,0,0,0,0
1,573f28f39b0ffc283676f277,"['messenger', 'facebook', 'dunzo', 'swiggy', '...",35,"['Ecommerce', 'Finance_Investment', 'News', 'S...",15,"['Ride haling', 'Rest', 'Finance_Investment', ...",4,1,1,1,0,0,0


### Customer app & cat explode mapping

In [10]:
df_customer_app_cat_mapping = pd.read_csv(local_datasets + 'processed/customer_app_categories_mapping.csv')
df_customer_cat_mapping = df_customer_app_cat_mapping[['customer_id', 
                                                       'categories_l1', 
                                                       'categories_l2']]\
                            .drop_duplicates()
df_customer_cat_mapping.head(1)

,customer_id,categories_l1,categories_l2
0,657bc9b5a55bd4f3ee494ef6,Tools,Rest


In [11]:
total_customers = df_customer_cat_mapping.customer_id.nunique()
total_customers

484284

In [12]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['Tools' 'Delivery_Food' 'Streaming_Music' 'Ecommerce' 'Social' 'OTT'
 'Travel_Ridehailing' 'Finance_Transactions' 'Finance_Investment' 'Office'
 'Educational' 'Travel_Bookings' 'Entertainment' 'Dating' 'News' 'Gaming'
 'Delivery_Grocery' 'Wearable' 'Health' 'Driver_App' 'Fantasy_Sports'
 'Finance_News' 'Devotional']


In [13]:
print(df_customer_app_cat_mapping['categories_l2'].unique())

['Rest' 'Ride haling' 'Finance_Investment' 'Office' 'Educational' 'Gaming'
 'Driver_App']


In [15]:
# single_category = ['Travel_Ridehailing']

# ### Office
# print(single_category)
# df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(single_category)]\
#             .groupby(['categories_l1','app_list'])\
#             .agg(customers = ('customer_id','nunique'))\
#             .sort_values(['customers'],ascending=False)\
#             .reset_index()
# df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
# df_temp

In [18]:
### Office
df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(['Ecommerce'])]\
            .groupby(['categories_l1','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l1', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,customers,%
0,Ecommerce,flipkart,279423,57.70
1,Ecommerce,meesho,172959,35.71
2,Ecommerce,myntra,118030,24.37
3,Ecommerce,ajio,98072,20.25
4,Ecommerce,jiomart,42936,8.87
5,Ecommerce,nykaa,39229,8.10
6,Ecommerce,firstcry,25385,5.24
7,Ecommerce,pharmeasy,10057,2.08
8,Ecommerce,zivame,6004,1.24
9,Ecommerce,amazon shopping,2821,0.58


### Merge raw data

In [27]:
df_customer_data = pd.merge(df_bangalore_active_customer[['customer_id', 
                                                          'service_affinity']],
                            df_customer_app_cat_mapping[['customer_id', 'app_name','categories_l1']]\
                                            [(df_customer_app_cat_mapping['categories_l1'].isin(['Ecommerce']))
                                             &
                                             (df_customer_app_cat_mapping['app_name'].isin(['flipkart']))
                                            ],
                            how='inner', on=['customer_id']
                           )
print(df_customer_data.shape)
df_customer_data.head(1)

(279423, 4)


,customer_id,service_affinity,app_name,categories_l1
0,657bc9b5a55bd4f3ee494ef6,ONLY_LINK,flipkart,Ecommerce


In [30]:
df_customer_data\
.groupby(['categories_l1', 'app_name', 'service_affinity'])\
.agg(customers = ('customer_id', 'nunique'))\
.reset_index()

,categories_l1,app_name,service_affinity,customers
0,Ecommerce,flipkart,AUTO_CAB,4594
1,Ecommerce,flipkart,LINK_AUTO,25896
2,Ecommerce,flipkart,LINK_CAB,1241
3,Ecommerce,flipkart,NO_AFFINITY,16404
4,Ecommerce,flipkart,ONLY_AUTO,109267
5,Ecommerce,flipkart,ONLY_CAB,5836
6,Ecommerce,flipkart,ONLY_LINK,102357
7,Ecommerce,flipkart,UNKNOWN,13828
